In [7]:
# Import libraries used for DICOM metadata scanning and parallel processing.
import pydicom
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Ask the user to choose the folder that holds the patient folders.
# Tries the Qt dialog first and falls back to tkinter if PyQt5 is not installed.
# Returns "" if the dialog is cancelled.
def select_folder(title="Select the folder containing your 4D flow patient folders"):
    try:
        from PyQt5.QtWidgets import QApplication, QFileDialog
        app = QApplication.instance() or QApplication([])   # reuse the app; do not call app.quit()
        return QFileDialog.getExistingDirectory(None, title)
    except ImportError:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)   # keeps the dialog from opening behind the notebook
        folder = filedialog.askdirectory(title=title)
        root.destroy()
        return folder

# Configure the source directory (chosen by the user) and series matching rule.
selected = select_folder()
if not selected:
    raise RuntimeError("No folder selected. Re-run this cell and choose a folder.")
main_folder = Path(selected)
print(f"Scanning: {main_folder}")
TARGET_SERIES_DESCRIPTION = "4D"
MATCH_MODE = "contains"

# Inspect one patient folder and collect matching scan metadata.
def process_patient(patient_dir):
    matches = []
    found_any = False
    for sequence_dir in sorted(patient_dir.iterdir()):
        if not sequence_dir.is_dir():
            continue
        dcm_files = list(sequence_dir.glob("*.dcm"))
        if not dcm_files:
            continue
        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            series_desc = getattr(ds, "SeriesDescription", "") or ""
            match = (
                TARGET_SERIES_DESCRIPTION.lower() in series_desc.lower()
                if MATCH_MODE == "contains"
                else series_desc.lower() == TARGET_SERIES_DESCRIPTION.lower()
            )
            if match:
                matches.append({
                    "Patient Name": getattr(ds, "PatientName", None),
                    "patient_folder": patient_dir.name,
                    "sequence_folder": sequence_dir.name,
                    "scan_path": str(sequence_dir),
                    "StudyDate": getattr(ds, "StudyDate", None),
                    "num_files": len(dcm_files),
                    "PatientMRN": getattr(ds, "PatientID", None),
                    "SeriesDescription": series_desc,
                    "Modality": getattr(ds, "Modality", None),
                })
                found_any = True
        except Exception as e:
            print(f"Skipped {sequence_dir}: {e}")
    return matches, (patient_dir.name if not found_any else None)

# Find patient folders and scan them concurrently.
patient_dirs = [d for d in main_folder.iterdir() if d.is_dir()]
records = []
no_match = []

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(process_patient, pd_dir): pd_dir for pd_dir in patient_dirs}
    for future in tqdm(as_completed(futures), total=len(futures)):
        matches, unmatched_patient = future.result()
        records.extend(matches)
        if unmatched_patient:
            no_match.append(unmatched_patient)

# Build the metadata table and rename columns for display.
df = pd.DataFrame(records)
print(f"Found {len(df)} matching sequences")
if df.empty:
    raise RuntimeError(
        "No sequences matched. Check that you chose the folder that contains the patient "
        f"folders, and that series descriptions contain '{TARGET_SERIES_DESCRIPTION}'."
    )

df = df.rename(columns={
    'patient_folder': 'Accession Number',
    'sequence_folder': 'Sequence Folder',
    'num_files': 'Image Count',
    'SeriesDescription' : 'Series Description',
})

# Apply table styling for notebook display.
(df.style
 .set_properties(**{
     'text-align': 'center',
     'font-family': 'Arial, sans-serif',
     'border': '1px solid white'
 }))

df["Patient Name"] = df["Patient Name"].astype(str)
df_sorted = df.sort_values("Patient Name").reset_index(drop=True)
print(f"Loaded {len(df_sorted)} rows")

Scanning: \\pom\pmcommon\_PM.Private\CSIC\Moulin\4D Flow Data


100%|██████████| 141/141 [07:48<00:00,  3.32s/it]

Found 410 matching sequences
Loaded 410 rows


In [9]:
# Enable interactive Matplotlib output for this notebook.
%matplotlib widget
%matplotlib inline

In [10]:
# Import widget, plotting, numerical, and canvas tools used by the viewer.
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Image as DisplayImage
from io import BytesIO
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from ipycanvas import Canvas, hold_canvas

def display_static_figure(fig):
    """Display a graph as a static image without Matplotlib's Save/Copy toolbar."""
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
    buffer.seek(0)
    display(DisplayImage(data=buffer.getvalue(), format="png"))
    buffer.close()

# Axis order for each anatomical view. Arrays are indexed [row, col, slice, vel, time],
# and the scans are assumed to be acquired as axial slices (rows = anterior-posterior,
# columns = left-right, slices = head-foot). The viewer shows array[:, :, slice, ...], so
# each view just permutes which anatomical direction ends up on the row / column / slice
# axis. Velocity (axis 3) and time (axis 4) never move. If your data is acquired in a
# different plane, edit this one table.
VIEW_PERMUTATIONS = {
    "Axial":    (0, 1, 2, 3, 4),
    "Coronal":  (2, 1, 0, 3, 4),
    "Sagittal": (2, 0, 1, 3, 4),
}

# Prepare display data and delineate visible and hidden columns.
df_display = df_sorted.reset_index(drop=True)
df_display["Patient Name"] = df_display["Patient Name"].str.replace("^", ", ", regex=False)
patient_cols = ["Patient Name", "PatientMRN"]
hidden_cols = ["scan_path", "StudyDate"]
scan_cols = [c for c in df_display.columns if c not in patient_cols + hidden_cols]

# Create reusable widget layouts and cell formatting.
cell_layout = widgets.Layout(width="150px", overflow="hidden")
checkbox_layout = widgets.Layout(width="60px")
row_layout = widgets.Layout(width="100%", min_height="35px", height="35px", padding="4px 0px", overflow="hidden")

# Create a centered HTML table cell with optional bold text.
def centered(text, bold=False):
    weight = "font-weight:bold;" if bold else ""
    return widgets.HTML(value=f"<div style='text-align:center; {weight}' title='{text}'>{text}</div>", layout=cell_layout)

# Create and style the patient search box.
search_box = widgets.Text(
    placeholder="Search by patient name or MRN...",
    layout=widgets.Layout(width="300px", margin="20px 0px 20px 10px")
)
search_box.add_class("mytext")

display(HTML("""
<style>
.mytext > .widget-label { font-size: 20px; }
.mytext > input[type="text"] {
    font-family: 'Calibri', sans-serif;
    font-size: 16px;
    border: 2px solid #36454F !important;
    border-radius: 4px;
}
</style>
"""))

# Build table headers and the selected-scan summary panel.
# Build the column header row for the scan-selection table.
def build_header():
    return widgets.HBox(
        [widgets.HTML(value="<div style='text-align:center; font-weight:bold;'>Select</div>", layout=checkbox_layout)] +
        [centered(col, bold=True) for col in scan_cols],
        layout=widgets.Layout(width="100%", border="0px 0px 2px 0px solid black", padding="4px 0px")
    )

selected_list_container = widgets.VBox(
    [],
    layout=widgets.Layout(display="flex", flex_flow="column", width="100%")
)

# Refresh the list of scans currently selected by the user.
def update_selected_summary():
    selected_indices = [i for i, cb in checkboxes.items() if cb.value]
    if not selected_indices:
        selected_list_container.children = [widgets.HTML(value="<i>No scans selected.</i>")]
        return

    rows = []
    for i in selected_indices:
        row = df_display.loc[i]
        seq_label = row.get("Sequence Folder", row.get(scan_cols[0], ""))
        label_text = f"{row['Patient Name']} | {seq_label}"
        label = widgets.HTML(
            value=f"<span style='font-size:13px;'>{label_text}</span>",
            layout=widgets.Layout(width="300px", overflow="hidden")
        )
        remove_btn = widgets.Button(
            description="x",
            layout=widgets.Layout(width="28px", height="28px"),
            tooltip="Remove"
        )

        # Create a callback that unchecks one selected scan.
        def make_remove_handler(idx):
            def handler(b):
                checkboxes[idx].value = False
            return handler

        remove_btn.on_click(make_remove_handler(i))
        rows.append(widgets.HBox(
            [label, remove_btn],
            layout=widgets.Layout(align_items="center", padding="2px 0px", min_height="30px", flex="0 0 auto")
        ))

    selected_list_container.children = rows

# Group scans by patient and build the accordion rows.
checkboxes = {}
patient_sections = {}

# Format DICOM study dates for readable display.
def format_study_date(date_val):
    try:
        return datetime.strptime(str(date_val), "%Y%m%d").strftime("%m/%d/%Y")
    except (ValueError, TypeError):
        return str(date_val)

# Extract the numeric sequence prefix used for scan ordering.
def scan_sort_key(sequence_folder):
    try:
        return int(str(sequence_folder).split("_")[0])
    except (ValueError, IndexError):
        return float("inf")

for patient_key, group in df_display.groupby(patient_cols, sort=False):
    name, mrn = patient_key if isinstance(patient_key, tuple) else (patient_key, "")
    title = f"{name} | MRN: {mrn} ({len(group)} scans)"
    rows = [build_header()]
    for date_val, date_group in group.sort_values("StudyDate", ascending=False).groupby("StudyDate", sort=False):
        rows.append(widgets.HTML(
            value=f"<div style='font-weight:bold; padding:8px 0 2px;'>{format_study_date(date_val)}</div>"
        ))
        date_group_sorted = date_group.assign(
            _scan_num=date_group["Sequence Folder"].apply(scan_sort_key)
        ).sort_values("_scan_num")
        for i, row in date_group_sorted.iterrows():
            cb = widgets.Checkbox(value=False, indent=False, layout=checkbox_layout)
            cb.observe(lambda change: update_selected_summary(), names="value")
            checkboxes[i] = cb
            cells = [cb] + [centered(row[col]) for col in scan_cols]
            rows.append(widgets.HBox(cells, layout=row_layout))

    patient_sections[patient_key] = (title, widgets.VBox(rows))

update_selected_summary()

# Configure the accordion and search filtering.
accordion = widgets.Accordion(children=[body for _, body in patient_sections.values()])
for idx, (title, _) in enumerate(patient_sections.values()):
    accordion.set_title(idx, title)

accordion_container = widgets.VBox(
    [accordion],
    layout=widgets.Layout(height="400px", width="100%", overflow="auto", border="1px solid lightgray")
)

# Replace accordion contents with the patients matching the search.
def rebuild_accordion(filtered_keys):
    children = [patient_sections[k][1] for k in filtered_keys]
    accordion.children = children
    for idx, k in enumerate(filtered_keys):
        accordion.set_title(idx, patient_sections[k][0])

# Filter patient sections when the search box changes.
def on_search_change(change):
    query = change["new"].strip().lower()
    if not query:
        rebuild_accordion(list(patient_sections.keys()))
    else:
        mask = (
            df_display["Patient Name"].astype(str).str.lower().str.contains(query, na=False) |
            df_display["PatientMRN"].astype(str).str.lower().str.contains(query, na=False)
        )
        matching_patients = df_display.loc[mask, patient_cols].drop_duplicates()
        filtered_keys = [tuple(x) for x in matching_patients.itertuples(index=False)]
        rebuild_accordion([k for k in patient_sections.keys() if k in filtered_keys])

search_box.observe(on_search_change, names="value")

# Create the run controls and output area.
selected_container = widgets.VBox(
    [selected_list_container],
    layout=widgets.Layout(height="100px", width="100%", overflow="auto", border="1px solid lightgray", padding="6px")
)
run_button = widgets.Button(description="Run", button_style="primary")
output = widgets.Output()

# Convert phase pixels and render magnitude or velocity images for the canvases.
# Convert encoded phase pixel values into signed velocity values.
def phase_to_velocity(phase_pixels, venc, grayscale_mid=2048):
    return (phase_pixels.astype(float) - grayscale_mid) / grayscale_mid * venc

# Overlay velocity colors on a grayscale magnitude image.
# Blend velocity colors over a normalized magnitude image.
def render_to_rgba(mag_img, vel_img, venc, alpha=0.5):
    mag_norm = (mag_img - mag_img.min()) / (mag_img.max() - mag_img.min() + 1e-8)
    gray_rgba = cm.gray(mag_norm)

    vel_norm = np.clip((vel_img + venc) / (2 * venc), 0, 1)
    vel_rgba = cm.seismic(vel_norm)

    out = gray_rgba.copy()
    out[..., :3] = (1 - alpha) * gray_rgba[..., :3] + alpha * vel_rgba[..., :3]
    out[..., 3] = 1.0
    return (out * 255).astype(np.uint8)

# Convert a magnitude image to grayscale RGBA pixels.
# Render a magnitude image as grayscale RGBA pixels.
def render_mag_only(mag_img):
    mag_norm = (mag_img - mag_img.min()) / (mag_img.max() - mag_img.min() + 1e-8)
    return (cm.gray(mag_norm) * 255).astype(np.uint8)

# Convert a globally-normalized MRA slice (0-1) to thresholded grayscale RGBA.
def render_mra_norm(norm_slice, threshold=0.0):
    rgba = (cm.gray(norm_slice) * 255).astype(np.uint8)
    rgba[norm_slice < threshold, :3] = 0
    return rgba

# Grow a 3D region from seed voxels through all touching (26-neighbour) voxels in mask.
# Pure numpy, so no scipy is needed.
_NEIGHBOR_OFFSETS = np.array(
    [(i, j, k) for i in (-1, 0, 1) for j in (-1, 0, 1) for k in (-1, 0, 1) if (i, j, k) != (0, 0, 0)]
)

def grow_region(mask, seeds):
    visited = seeds.copy()
    frontier = np.argwhere(seeds)
    shape = np.array(mask.shape)
    while len(frontier):
        nb = (frontier[:, None, :] + _NEIGHBOR_OFFSETS[None, :, :]).reshape(-1, 3)
        nb = nb[np.all((nb >= 0) & (nb < shape), axis=1)]
        nb = nb[mask[nb[:, 0], nb[:, 1], nb[:, 2]] & ~visited[nb[:, 0], nb[:, 1], nb[:, 2]]]
        if len(nb) == 0:
            break
        nb = np.unique(nb, axis=0)
        visited[nb[:, 0], nb[:, 1], nb[:, 2]] = True
        frontier = nb
    return visited

# Manage image display and line drawing for one interactive canvas.
IMAGE_DISPLAY_SCALE = 2
class CanvasLineDrawer:
    """Wraps an ipycanvas Canvas: shows an image, lets the user click-drag-release to draw lines."""
    # Initialize the canvas and mouse handlers for line drawing.
    def __init__(self, image_rgba):
        h, w = image_rgba.shape[:2]
        self.scale = IMAGE_DISPLAY_SCALE
        self.canvas = Canvas(width=w * self.scale, height=h * self.scale)
        self.image_rgba = image_rgba
        self.lines = []  # (xmin, ymin, xmax, ymax) per completed line
        self.start = None
        self.on_line_drawn = None
        self._mra_missing = None  # set externally for the MRA panel when directions are missing

        self.draw_image()

        self.canvas.on_mouse_down(self.on_press)
        self.canvas.on_mouse_move(self.on_motion)
        self.canvas.on_mouse_up(self.on_release)

    # Redraw the base image and all completed lines.
    def draw_image(self):
        display_image = self.image_rgba.repeat(self.scale, axis=0).repeat(self.scale, axis=1)
        with hold_canvas(self.canvas):
            self.canvas.put_image_data(display_image, 0, 0)
            self.canvas.stroke_style = "yellow"
            self.canvas.line_width = 2
            for (x0, y0, x1, y1) in self.lines:
                self.canvas.stroke_line(
                    x0 * self.scale, y0 * self.scale, x1 * self.scale, y1 * self.scale
                )

    # Replace the displayed image while preserving drawn lines.
    # The canvas is resized to the new image, because a coronal or sagittal view
    # has a different shape from the axial one.
    def set_image(self, new_image_rgba):
        self.image_rgba = new_image_rgba
        h, w = new_image_rgba.shape[:2]
        self.canvas.width = w * self.scale
        self.canvas.height = h * self.scale
        self.draw_image()

    # Record the starting point of a new line.
    def on_press(self, x, y):
        self.start = (x / self.scale, y / self.scale)

    # Preview the line while the mouse is being dragged.
    def on_motion(self, x, y):
        if self.start is None:
            return
        x0, y0 = self.start
        x /= self.scale
        y /= self.scale
        with hold_canvas(self.canvas):
            display_image = self.image_rgba.repeat(self.scale, axis=0).repeat(self.scale, axis=1)
            self.canvas.put_image_data(display_image, 0, 0)
            self.canvas.stroke_style = "yellow"
            self.canvas.line_width = 2
            for (lx0, ly0, lx1, ly1) in self.lines:
                self.canvas.stroke_line(
                    lx0 * self.scale, ly0 * self.scale, lx1 * self.scale, ly1 * self.scale
                )
            self.canvas.line_width = 1
            self.canvas.stroke_line(
                x0 * self.scale, y0 * self.scale, x * self.scale, y * self.scale
            )

    # Reserve a hook for future line-vector calculations.
    def calculate_vector(self, lines):
        if not lines:
            return
        # Placeholder - fill in with real vector/length calculations when needed.

    # Store a completed line and notify the profile plot.
    def on_release(self, x, y):
        if self.start is None:
            return
        x0, y0 = self.start
        self.lines.append((x0, y0, x / self.scale, y / self.scale))
        self.start = None
        self.draw_image()
        self.calculate_vector(self.lines)
        if self.on_line_drawn is not None:
            self.on_line_drawn()

    # Remove all lines from the current canvas.
    def clear_lines(self):
        self.lines.clear()
        self.draw_image()
        if self.on_line_drawn is not None:
            self.on_line_drawn()

# Display magnitude, each velocity direction, and MRA in separate tabs.
# Build and connect the interactive scan viewer for selected scans.
def run_function(scans):
    # Work on a local copy of the scan dictionaries so that changing the view
    # (which swaps each scan's "array" for a transposed view) never alters the caller's data.
    scans = {idx: dict(s) for idx, s in scans.items()}
    original_arrays = {idx: s["array"] for idx, s in scans.items()}


    mag_idxs = [idx for idx, s in scans.items() if s["type"] == "M"]
    phase_idxs = [idx for idx, s in scans.items() if s["type"] == "P"]

    if not mag_idxs or not phase_idxs:
        print("Need at least one magnitude and one phase scan selected.")
        return

    mag_arr = scans[mag_idxs[0]]["array"]
    n_slices, n_time = mag_arr.shape[2], mag_arr.shape[4]
    v_mag = first_nonzero_vel(mag_arr)

    # Time frame 0 is an empty placeholder (phase 0 decodes to exactly -venc everywhere),
    # so every calculation starts at frame 1, matching the time slider.
    FIRST_FRAME = 1

    direction_panels = {}
    unlabeled = []
    for phase_idx in phase_idxs:
        for v, label in scans[phase_idx]["vel_directions"].items():
            if label:
                direction_panels.setdefault(label, (phase_idx, v))
            else:
                unlabeled.append((phase_idx, v))

    # --- DEBUG OVERRIDE: assume ordered X, Y, Z when no real labels were found ---
    DEBUG_ASSUME_XYZ_ORDER = True  # set False to go back to real label detection only

    if DEBUG_ASSUME_XYZ_ORDER and not direction_panels:
        all_vel_entries = []
        for phase_idx in phase_idxs:
            for v in scans[phase_idx]["vel_directions"].keys():
                all_vel_entries.append((phase_idx, v))
        all_vel_entries.sort(key=lambda pv: (pv[0], pv[1]))  # stable order: by phase_idx, then vel index

        for axis, entry in zip(["X", "Y", "Z"], all_vel_entries):
            direction_panels[axis] = entry
        unlabeled = all_vel_entries[3:]  # anything beyond the first 3 stays unlabeled
    # --- end debug override ---

    ordered_labels = [lbl for lbl in ["X", "Y", "Z"] if lbl in direction_panels]
    ordered_labels += [lbl for lbl in direction_panels if lbl not in ordered_labels]
    velocity_panels = [
        (f"V{lbl.lower()}" if lbl in ["X", "Y", "Z"] else lbl, direction_panels[lbl])
        for lbl in ordered_labels
    ]
    velocity_panels += [(f"Scan {p}, vel {v}", (p, v)) for p, v in unlabeled]
    if not velocity_panels:
        print("Selected phase scans do not contain any velocity channels.")
        return
    if n_time <= FIRST_FRAME:
        print("Selected scans do not contain a usable time frame.")
        return

    panel_names = ["Vmag"] + [lbl for lbl, _ in velocity_panels] + ["MRA"]

    view_dropdown = widgets.Dropdown(
        options=list(VIEW_PERMUTATIONS.keys()), value="Axial", description="View:"
    )
    slice_slider = widgets.IntSlider(min=0, max=n_slices - 1, value=n_slices // 2, description="Slice:", continuous_update=False)
    time_slider = widgets.IntSlider(min=FIRST_FRAME, max=n_time - 1, value=1, description="Time:", continuous_update=False)
    venc_box = widgets.FloatText(value=scans[velocity_panels[0][1][0]]["venc"] or 150.0, description="Venc (cm/s):")
    mra_threshold_box = widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=1.0,
        step=0.01,
        description="MRA threshold:",
        readout_format=".2f",
        continuous_update=False,
    )
    clear_lines_button = widgets.Button(description="Clear mask")

    # Colorbar height matches the displayed image height, with a minimum so labels stay readable.
    def colorbar_height():
        return max(120, mag_arr.shape[0] * IMAGE_DISPLAY_SCALE)

    # Render the velocity colorbar used beside velocity panels.
    def render_colorbar(venc, height, width=80):
        fig, ax = plt.subplots(figsize=(width / 100, height / 100), dpi=100)
        gradient = np.linspace(venc, -venc, height).reshape(-1, 1)
        ax.imshow(gradient, aspect="auto", cmap="seismic", vmin=-venc, vmax=venc,
                extent=[0, 1, -venc, venc])
        ax.set_xticks([])
        ax.yaxis.tick_right()
        ax.tick_params(labelsize=8)
        ax.set_ylabel("Velocity (cm/s)", rotation=270, labelpad=15, fontsize=8)
        ax.yaxis.set_label_position("right")
        fig.subplots_adjust(left=0.05, right=0.55, top=0.95, bottom=0.05)

        fig.canvas.draw()
        buf = np.asarray(fig.canvas.buffer_rgba())
        plt.close(fig)
        return buf

    # Cache the 3D normalized MRA volume (H, W, slices), one per (venc, view), because
    # the volume's shape changes with the view.
    # MRA = (1/N) * sum over time of [ M^2 * (Vx^2 + Vy^2 + Vz^2) ], normalized over the whole volume.
    mra_cache = {}

    def get_mra_volume(venc):
        missing = [a for a in ["X", "Y", "Z"] if a not in direction_panels]
        if missing:
            return None, missing
        cache_key = (venc, view_dropdown.value)
        if cache_key in mra_cache:
            return mra_cache[cache_key], None

        H, W, S = mag_arr.shape[:3]
        accum = np.zeros((H, W, S), dtype=float)
        for t in range(FIRST_FRAME, n_time):
            M = mag_arr[:, :, :, v_mag, t].astype(float)
            v_sum_sq = np.zeros_like(M)
            for axis in ["X", "Y", "Z"]:
                phase_idx, v_phase = direction_panels[axis]
                p = scans[phase_idx]["array"][:, :, :, v_phase, t]
                v_sum_sq += phase_to_velocity(p, venc) ** 2
            accum += (M ** 2) * v_sum_sq

        disp = np.sqrt(np.clip(accum / (n_time - FIRST_FRAME), 0, None))
        norm = (disp - disp.min()) / (disp.max() - disp.min() + 1e-8)
        mra_cache[cache_key] = norm
        return norm, None

    mask_curve_output = widgets.Output(
        layout=widgets.Layout(width="340px", min_height="260px", border="1px solid lightgray")
    )

    # The slice each drawn line was drawn on (parallel to the shared list of drawn lines).
    mask_state = {"seed_slices": []}
    vessel_cache = {}
    graph_data_cache = {}

    # Pixels (row, col) under a drawn line.
    def line_seed_pixels(line, shape_hw):
        x0, y0, x1, y1 = line
        height, width = shape_hw
        sample_count = max(2, int(np.ceil(np.hypot(x1 - x0, y1 - y0))) + 1)
        xs = np.linspace(x0, x1, sample_count)
        ys = np.linspace(y0, y1, sample_count)
        cols = np.clip(np.rint(xs).astype(int), 0, width - 1)
        rows = np.clip(np.rint(ys).astype(int), 0, height - 1)
        return rows, cols

    # Vessel mask for drawn line k: the voxels above the MRA threshold that are connected
    # in 3D to the line. Only the slices this vessel passes through are included.
    # Returns (vessel, None) on success or (None, message) if it cannot be built.
    def get_vessel(k):
        venc, thr = venc_box.value, mra_threshold_box.value
        norm, missing = get_mra_volume(venc)
        if norm is None:
            return None, f"MRA mask requires X, Y, Z directions. Missing: {', '.join(missing)}"
        if thr <= 0:
            return None, "Raise the MRA threshold above 0 so the vessel can be separated from background."

        lines, seed_slices = drawers["MRA"].lines, mask_state["seed_slices"]
        if k >= len(lines) or k >= len(seed_slices):
            return None, "Draw a line through the vessel to calculate its velocity curves."
        line, seed_slice = tuple(lines[k]), seed_slices[k]

        key = (line, seed_slice, thr, venc, view_dropdown.value)
        if key in vessel_cache:
            return vessel_cache[key], None

        rows, cols = line_seed_pixels(line, norm.shape[:2])
        mask = norm >= thr                                       # (H, W, S)
        seeds = np.zeros_like(mask)
        seeds[rows, cols, seed_slice] = True
        seeds &= mask
        if not seeds.any():
            return None, (f"Line {k + 1} does not cross any voxels above the MRA threshold. "
                          "Lower the threshold or redraw the line through the vessel.")

        vessel = grow_region(mask, seeds)
        if len(vessel_cache) > 20:
            vessel_cache.clear()
        vessel_cache[key] = vessel
        return vessel, None

    # Velocity (cm/s) of one channel for every vessel voxel: shape (N_voxels, T).
    def vessel_component(vessel, venc, phase_idx, v_phase):
        raw = scans[phase_idx]["array"][:, :, :, v_phase, FIRST_FRAME:][vessel, :]
        return phase_to_velocity(raw, venc)

    # Velocity vectors for every vessel voxel: shape (3, N_voxels, T) for X, Y, Z.
    def vessel_vectors(vessel, venc):
        return np.stack([
            vessel_component(vessel, venc, *direction_panels[axis]) for axis in ["X", "Y", "Z"]
        ])

    # Plot the along-vessel curve (plus Vx, Vy, Vz) for the most recently drawn line.
    def update_mask_curves():
        mask_curve_output.clear_output(wait=True)
        graph_data_cache.pop("MRA (along vessel)", None)
        venc = venc_box.value
        with mask_curve_output:
            lines = drawers["MRA"].lines
            if not lines:
                print("Draw a line through the vessel to calculate its velocity curves.")
                return
            vessel, message = get_vessel(len(lines) - 1)
            if vessel is None:
                print(message)
                return
            graph_data_cache["MRA (along vessel)"] = []

            slices_used = np.where(vessel.any(axis=(0, 1)))[0]
            n_vox = int(vessel.sum())
            V = vessel_vectors(vessel, venc)                        # (3, N, T)

            # Along-vessel velocity: project each voxel's velocity onto its own mean flow
            # direction, then average with weights = time-averaged speed. This avoids the
            # cancellation that plain Vx/Vy/Vz means suffer when the vein bends, and
            # down-weights noisy voxels with almost no net flow.
            vbar = V.mean(axis=2)                                   # (3, N)
            speed = np.linalg.norm(vbar, axis=0)                    # (N,)
            unit = vbar / (speed + 1e-8)
            projected = np.einsum("cn,cnt->nt", unit, V)            # (N, T)
            along = (projected * speed[:, None]).sum(axis=0) / (speed.sum() + 1e-8)
            graph_data_cache["MRA (along vessel)"] = [
                {"panel": "MRA", "series": "Vx", "time_frame": int(t), "velocity_cm_s": float(v)}
                for t, v in zip(np.arange(FIRST_FRAME, n_time), V[0].mean(axis=0))
            ] + [
                {"panel": "MRA", "series": "Vy", "time_frame": int(t), "velocity_cm_s": float(v)}
                for t, v in zip(np.arange(FIRST_FRAME, n_time), V[1].mean(axis=0))
            ] + [
                {"panel": "MRA", "series": "Vz", "time_frame": int(t), "velocity_cm_s": float(v)}
                for t, v in zip(np.arange(FIRST_FRAME, n_time), V[2].mean(axis=0))
            ] + [
                {"panel": "MRA", "series": "Along vessel", "time_frame": int(t), "velocity_cm_s": float(v)}
                for t, v in zip(np.arange(FIRST_FRAME, n_time), along)
            ]

            fig, ax = plt.subplots(figsize=(3.2, 2.4), dpi=100)
            t_axis = np.arange(FIRST_FRAME, n_time)
            colors = {"X": "tab:red", "Y": "tab:green", "Z": "tab:blue"}
            for i, axis in enumerate(["X", "Y", "Z"]):
                ax.plot(t_axis, V[i].mean(axis=0), color=colors[axis], linewidth=1,
                        alpha=0.6, label=f"V{axis.lower()}")
            ax.plot(t_axis, along, color="black", linewidth=2.5, label="Along vessel")
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_title(
                f"Vessel velocity: slices {slices_used.min()}-{slices_used.max()} "
                f"({len(slices_used)} slices, {n_vox} voxels)"
            )
            ax.set_xlabel("Time frame")
            ax.set_ylabel("Velocity (cm/s)")
            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()
            display_static_figure(fig)
            plt.close(fig)

    # Create one time-frame velocity overlay for a selected direction.
    def make_velocity_image(phase_idx, v_phase):
        s, t, venc = slice_slider.value, time_slider.value, venc_box.value
        mag_img = mag_arr[:, :, s, v_mag, t]
        vel_img = phase_to_velocity(scans[phase_idx]["array"][:, :, s, v_phase, t], venc)
        return render_to_rgba(mag_img, vel_img, venc)

    # Per-tab plot: for each drawn line, the mean velocity over time across the WHOLE
    # vessel mask (every voxel of the vessel in every through-plane slice it crosses),
    # not just the pixels under the line.
    #   Vx / Vy / Vz tabs -> mean of that component inside the vessel
    #   Vmag / MRA tabs   -> mean speed |V| inside the vessel
    def update_profile(view_name):
        profile_outputs[view_name].clear_output(wait=True)
        graph_data_cache.pop(view_name, None)
        lines = drawers[view_name].lines
        if not lines:
            return

        venc = venc_box.value
        channel = dict(velocity_panels).get(view_name)            # None for Vmag / MRA
        label = view_name if channel is not None else "Speed |V|"
        t_axis = np.arange(FIRST_FRAME, n_time)

        with profile_outputs[view_name]:
            fig, ax = plt.subplots(figsize=(3.2, 2.4), dpi=100)
            notes, plotted = [], False
            graph_data_cache[view_name] = []
            for k in range(len(lines)):
                vessel, message = get_vessel(k)
                if vessel is None:
                    notes.append(message)
                    continue
                if channel is not None:
                    curve = vessel_component(vessel, venc, *channel).mean(axis=0)
                else:
                    V = vessel_vectors(vessel, venc)
                    curve = np.sqrt((V ** 2).sum(axis=0)).mean(axis=0)
                slices_used = np.where(vessel.any(axis=(0, 1)))[0]
                ax.plot(t_axis, curve,
                        label=f"Line {k + 1}: slices {slices_used.min()}-{slices_used.max()}, "
                              f"{int(vessel.sum())} vox")
                plotted = True
                series_name = f"Line {k + 1}"
                graph_data_cache.setdefault(view_name, [])
                graph_data_cache[view_name] = [
                    row for row in graph_data_cache[view_name]
                    if row["series"] != series_name
                ] + [
                    {"panel": view_name, "series": series_name, "time_frame": int(t), "velocity_cm_s": float(v)}
                    for t, v in zip(t_axis, curve)
                ]

            if plotted:
                ax.axhline(0, color="black", linewidth=0.8)
                ax.set_title(f"{label} over time (inside vessel mask)")
                ax.set_xlabel("Time frame")
                ax.set_ylabel("Velocity (cm/s)")
                ax.grid(True, alpha=0.3)
                ax.legend(fontsize=7)
                fig.tight_layout()
                display_static_figure(fig)
            plt.close(fig)
            for note in dict.fromkeys(notes):
                print(note)

    def export_graph_data(file_type, status_output):
        rows = [row for panel_rows in graph_data_cache.values() for row in panel_rows]
        if not rows:
            with status_output:
                status_output.clear_output()
                print("Draw a vessel line first so graph data is available to export.")
            return

        export_df = pd.DataFrame(rows).sort_values(["panel", "series", "time_frame"])
        output_path = Path.cwd() / f"4dflow_graph_data.{file_type}"
        with status_output:
            status_output.clear_output()
            if file_type == "csv":
                export_df.to_csv(output_path, index=False)
            elif file_type == "xlsx":
                export_df.to_excel(output_path, index=False, sheet_name="Graph data")
            else:
                raise ValueError(f"Unsupported export type: {file_type}")
            display(widgets.HTML(
                f"<b>Exported:</b> <code>{output_path}</code>"
            ))

    # Build one drawer per panel up front.
    drawers = {}
    colorbars = {}

    # Magnitude panel.
    mag_img0 = mag_arr[:, :, slice_slider.value, v_mag, time_slider.value]
    drawers["Vmag"] = CanvasLineDrawer(render_mag_only(mag_img0))
    colorbars["Vmag"] = None

    # Velocity direction panels.
    for label, (phase_idx, v_phase) in velocity_panels:
        drawers[label] = CanvasLineDrawer(make_velocity_image(phase_idx, v_phase))
        cbar_img = render_colorbar(venc_box.value, height=colorbar_height())
        cbar_canvas = Canvas(width=cbar_img.shape[1], height=cbar_img.shape[0])
        cbar_canvas.put_image_data(cbar_img, 0, 0)
        colorbars[label] = cbar_canvas

    # MRA panel.
    mra_norm, missing = get_mra_volume(venc_box.value)
    if mra_norm is None:
        placeholder = np.zeros((mag_arr.shape[0], mag_arr.shape[1], 4), dtype=np.uint8)
        drawers["MRA"] = CanvasLineDrawer(placeholder)
        drawers["MRA"]._mra_missing = missing
    else:
        drawers["MRA"] = CanvasLineDrawer(
            render_mra_norm(mra_norm[:, :, slice_slider.value], mra_threshold_box.value)
        )
        drawers["MRA"]._mra_missing = None
    colorbars["MRA"] = None

    # Create one profile output per tab and keep one shared drawn mask.
    profile_outputs = {
        name: widgets.Output(
        layout=widgets.Layout(width="340px", min_height="260px", border="1px solid lightgray")
        )
        for name in panel_names
    }

    def synchronize_mask(source_name):
        shared_lines = list(drawers[source_name].lines)
        for name, drawer in drawers.items():
            if name != source_name:
                drawer.lines = list(shared_lines)
                drawer.draw_image()
        for name in panel_names:
            update_profile(name)

    def make_mask_handler(name):
        def handler():
            # A line was just drawn (or cleared) on this tab at the current slice.
            lines, seeds = drawers[name].lines, mask_state["seed_slices"]
            if not lines:
                seeds.clear()
            elif len(lines) > len(seeds):
                seeds.append(slice_slider.value)
            synchronize_mask(name)
            update_mask_curves()
        return handler

    for name, drawer in drawers.items():
        drawer.on_line_drawn = make_mask_handler(name)

    tab_contents = []
    for name in panel_names:
        drawer = drawers[name]
        if name == "MRA" and drawer._mra_missing:
            tab_contents.append(widgets.HTML(
                value=f"<i>MRA requires X, Y, Z directions. Missing: {', '.join(drawer._mra_missing)}</i>"
            ))
            continue
        scan_widgets = [drawer.canvas]
        if colorbars[name] is not None:
            scan_widgets.append(colorbars[name])
        image_panel = widgets.HBox(
            scan_widgets,
            layout=widgets.Layout(align_items="flex-start", width="fit-content"),
        )
        graph_panel = widgets.VBox(
            layout=widgets.Layout(width="360px", align_items="stretch", margin="0 0 0 12px")
        )
        export_csv_button = widgets.Button(description="Export graph (CSV)")
        export_excel_button = widgets.Button(description="Export graph (Excel)")
        export_status = widgets.Output()
        export_csv_button.on_click(lambda b, status=export_status: export_graph_data("csv", status))
        export_excel_button.on_click(lambda b, status=export_status: export_graph_data("xlsx", status))
        export_controls = widgets.VBox([
            widgets.HBox([export_csv_button, export_excel_button]),
            export_status,
        ])
        if name == "MRA":
            graph_panel.children = (mask_curve_output, profile_outputs[name], export_controls)
        else:
            graph_panel.children = (profile_outputs[name], export_controls)
        tab_contents.append(
            widgets.HBox(
                [image_panel, graph_panel],
                layout=widgets.Layout(align_items="flex-start"),
            )
        )
    view_tabs = widgets.Tab(children=tab_contents)
    for index, name in enumerate(panel_names):
        view_tabs.set_title(index, name)

    # Recompute images and profiles when a viewer control changes.
    def update_all(change=None):
        s, t, venc = slice_slider.value, time_slider.value, venc_box.value

        drawers["Vmag"].set_image(render_mag_only(mag_arr[:, :, s, v_mag, t]))

        for label, (phase_idx, v_phase) in velocity_panels:
            drawers[label].set_image(make_velocity_image(phase_idx, v_phase))
            # The colorbar must match the displayed height of the current view.
            cbar_img = render_colorbar(venc, height=colorbar_height())
            colorbars[label].width = cbar_img.shape[1]
            colorbars[label].height = cbar_img.shape[0]
            colorbars[label].put_image_data(cbar_img, 0, 0)

        norm, missing = get_mra_volume(venc)
        if norm is not None:
            drawers["MRA"].set_image(render_mra_norm(norm[:, :, s], mra_threshold_box.value))
        else:
            drawers["MRA"].set_image(np.zeros((*mag_arr.shape[:2], 4), dtype=np.uint8))
        drawers["MRA"]._mra_missing = missing

        # The vessel curves cover all slices and times, so only redo them when the mask
        # changes (venc or threshold), not when the slice or time sliders move.
        if change is None or change["owner"] in (venc_box, mra_threshold_box):
            for name in panel_names:
                update_profile(name)
            update_mask_curves()

    # Switch between axial, coronal and sagittal by permuting the axes of every scan array.
    # transpose() returns a view, so nothing is copied.
    def on_view_change(change):
        nonlocal mag_arr
        perm = VIEW_PERMUTATIONS[view_dropdown.value]
        for idx, original in original_arrays.items():
            scans[idx]["array"] = original.transpose(perm)
        mag_arr = scans[mag_idxs[0]]["array"]

        # Lines, seed slices and vessels drawn in the previous plane no longer apply.
        for drawer in drawers.values():
            drawer.lines.clear()
            drawer.start = None
        mask_state["seed_slices"].clear()
        vessel_cache.clear()

        # Re-range the slice slider for the new plane (observer paused so we update once).
        slice_slider.unobserve(update_all, names="value")
        slice_slider.max = mag_arr.shape[2] - 1
        slice_slider.value = mag_arr.shape[2] // 2
        slice_slider.observe(update_all, names="value")
        update_all()

    # Clear the shared mask and its profiles from every tab.
    def on_clear_lines(b):
        for drawer in drawers.values():
            drawer.lines.clear()
            drawer.start = None
            drawer.draw_image()
        mask_state["seed_slices"].clear()
        for name in panel_names:
            update_profile(name)
        update_mask_curves()

    view_dropdown.observe(on_view_change, names="value")
    slice_slider.observe(update_all, names="value")
    time_slider.observe(update_all, names="value")
    venc_box.observe(update_all, names="value")
    mra_threshold_box.observe(update_all, names="value")
    clear_lines_button.on_click(on_clear_lines)

    display(widgets.VBox([
        view_tabs,
        view_dropdown,
        slice_slider,
        time_slider,
        venc_box,
        mra_threshold_box,
        clear_lines_button,
    ]))

    update_mask_curves()
    return drawers

# Load the selected scans when the Run button is pressed.
def on_run_click(b):
    with output:
        clear_output()
        selected_indices = [i for i, cb in checkboxes.items() if cb.value]
        if not selected_indices:
            print("No scans selected.")
            return
        triage(df_display.loc[selected_indices])

run_button.on_click(on_run_click)

# Assemble and display the complete interactive viewer.
table = widgets.VBox([
    search_box,
    accordion_container,
    widgets.HTML(value="<b>Selected scans</b>"),
    selected_container,
    run_button,
    output,
])
display(table)

In [11]:
# Import DICOM, array, path, and Siemens metadata helpers for scan loading.
import pydicom
import numpy as np
from pathlib import Path
from nibabel.nicom import csareader

# Identify the dataframe column containing each scan folder.
SCAN_PATH_COL = "scan_path"

# Return the DICOM files in a stable order.
def get_dicom_files(folder):
    return sorted(Path(folder).glob("*.dcm"))

# Read one DICOM series and reconstruct its five-dimensional image array,
# capturing image type, venc, and per-direction labels in the same pass.
def load_dicom_series(folder):
    """Read a series once and return its array, image type, venc, and per-vel direction labels."""
    files = get_dicom_files(folder)
    if not files:
        raise ValueError(f"No DICOM files found in {folder}")

    # Map Siemens ICE_Dims positions to time, velocity, and slice axes.
    dim_time = 3 - 1
    dim_phase = 4 - 1
    dim_slice = 7 - 1

    images = []
    ice_vals_list = []
    skipped = []
    image_type = None
    venc = None
    vel_directions = {}  # vel index -> "X"/"Y"/"Z" label (or None if not found)

    # Read each file once, extracting metadata and pixels together.
    for file_index, fname in enumerate(files):
        dcm = pydicom.dcmread(fname)

        if file_index == 0:
            dcm_image_type = getattr(dcm, "ImageType", [])
            if "P" in dcm_image_type:
                image_type = "P"
            elif "M" in dcm_image_type:
                image_type = "M"

        csa = csareader.get_csa_header(dcm, "image")
        if csa is None:
            skipped.append(fname.name)
            continue

        if venc is None:
            venc_items = csa["tags"].get("FlowVenc", {}).get("items", [])
            if venc_items:
                venc = float(venc_items[0])

        ice_raw = csa["tags"]["ICE_Dims"]["items"][0]
        vals = [int(p.strip()) if p.strip().lstrip("-").isdigit() else 0 for p in ice_raw.split("_")]

        v = vals[dim_phase]
        if v not in vel_directions:
            dir_items = csa["tags"].get("FlowEncodingDirectionString", {}).get("items", [])
            vel_directions[v] = str(dir_items[0]).upper() if dir_items else None

        images.append(dcm.pixel_array.astype(float))
        ice_vals_list.append(vals)

    # Validate the decoded series before allocating the output array.
    if skipped:
        print(f"[warn] skipped {len(skipped)} file(s) with no CSA header in {folder}")
    if not images:
        raise ValueError(f"No valid CSA headers found in {folder}")
    if image_type is None:
        raise ValueError(f"Could not determine ImageType in {folder}")

    # Allocate the reconstructed array and place each image by ICE_Dims.
    ice_vals = np.asarray(ice_vals_list)
    n_slices = int(ice_vals[:, dim_slice].max()) + 1
    n_times = int(ice_vals[:, dim_time].max()) + 1
    n_vel = int(ice_vals[:, dim_phase].max()) + 1
    rows, cols = images[0].shape

    dcm_array = np.zeros((rows, cols, n_slices, n_vel, n_times), dtype=float)
    for image, vals in zip(images, ice_vals):
        s, t, v = vals[dim_slice], vals[dim_time], vals[dim_phase]
        dcm_array[:, :, s, v, t] = image

    return dcm_array, image_type, venc, vel_directions

# Find the first populated velocity channel efficiently.
def first_nonzero_vel(scan):
    nonzero_velocities = np.any(scan != 0, axis=(0, 1, 2, 4))
    nonzero_indices = np.flatnonzero(nonzero_velocities)
    return int(nonzero_indices[0]) if nonzero_indices.size else 0

# Cache decoded arrays so repeated Run-button clicks avoid network reads.
_scan_cache = {}

# Load selected scans, reusing cached data when possible.
def triage(selected_df):
    """Load each selected scan once, using cached arrays when available."""
    scans = {}
    total = len(selected_df)

    for position, (idx, row) in enumerate(selected_df.iterrows(), start=1):
        folder = str(row[SCAN_PATH_COL])
        if folder in _scan_cache:
            arr, img_type, venc, vel_directions = _scan_cache[folder]
            print(f"Using cached scan {position}/{total}: {Path(folder).name}")
        else:
            print(f"Loading scan {position}/{total}: {Path(folder).name}")
            arr, img_type, venc, vel_directions = load_dicom_series(folder)
            _scan_cache[folder] = (arr, img_type, venc, vel_directions)

        scans[idx] = {"array": arr, "type": img_type, "venc": venc, "folder": folder, "vel_directions": vel_directions}

    # Start the interactive viewer after all selected scans are ready.
    if not scans:
        print("No usable scans loaded.")
        return

    run_function(scans)